In [1]:
import os

SRC_DIR = "/kaggle/input/datasets/swetha344/pyg-210-cu128-wheels"
DST_DIR = "/kaggle/working/fixed_wheels"

os.makedirs(DST_DIR, exist_ok=True)

rename_map = {
    "torch_scatter-2.1.2pt210cu128-cp312-cp312-linux_x86_64.whl": "torch_scatter-2.1.2+pt210cu128-cp312-cp312-linux_x86_64.whl",
    "torch_cluster-1.6.3pt210cu128-cp312-cp312-linux_x86_64.whl": "torch_cluster-1.6.3+pt210cu128-cp312-cp312-linux_x86_64.whl",
    "pyg_lib-0.4.0pt210cu128-cp312-cp312-linux_x86_64.whl": "pyg_lib-0.4.0+pt210cu128-cp312-cp312-linux_x86_64.whl",
    "torch_geometric-2.7.0-py3-none-any.whl": "torch_geometric-2.7.0-py3-none-any.whl",
    "ogb-1.3.6-py3-none-any.whl": "ogb-1.3.6-py3-none-any.whl"
}

for old_name, new_name in rename_map.items():
    src = os.path.join(SRC_DIR, old_name)
    dst = os.path.join(DST_DIR, new_name)
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)

!pip install --no-index --no-deps --force-reinstall /kaggle/working/fixed_wheels/*.whl

Processing ./fixed_wheels/ogb-1.3.6-py3-none-any.whl
Processing ./fixed_wheels/torch_cluster-1.6.3+pt210cu128-cp312-cp312-linux_x86_64.whl
Processing ./fixed_wheels/torch_geometric-2.7.0-py3-none-any.whl
Processing ./fixed_wheels/torch_scatter-2.1.2+pt210cu128-cp312-cp312-linux_x86_64.whl


In [2]:
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

In [3]:
import torch
import torch_geometric

# We must use the exact internal paths PyG uses
from torch_geometric.data.data import DataEdgeAttr
from torch_geometric.data.storage import NodeStorage, EdgeStorage, GlobalStorage

# Register them globally with PyTorch's security layer
torch.serialization.add_safe_globals([
    DataEdgeAttr, 
    NodeStorage, 
    EdgeStorage, 
    GlobalStorage,
    torch_geometric.data.Data  # Sometimes needed for the base class
])

# NOW import and run OGB
from ogb.nodeproppred import PygNodePropPredDataset
import torch_geometric.transforms as T

with torch.serialization.safe_globals([DataEdgeAttr, NodeStorage, EdgeStorage]):
    dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='dataset/', transform=T.ToUndirected())

Downloaded 0.08 GB: 100%|██████████| 81/81 [00:03<00:00, 23.27it/s]
Processing...


Extracting dataset/arxiv.zip
Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:00<00:00, 9489.38it/s]


Converting graphs into PyG objects...


100%|██████████| 1/1 [00:00<00:00, 225.97it/s]

Saving...



Done!
/usr/local/lib/python3.12/dist-packages/ogb/nodeproppred/dataset_pyg.py:69: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  self.data, self.slices = torch.load(self.processed_paths[0])


In [4]:
# 1. Basic Dataset Info
print(f"Number of graphs: {len(dataset)}")
print(f"Number of classes: {dataset.num_classes}")
print(f"Number of node features: {dataset.num_node_features}")

# 2. Get the actual Data object
data = dataset[0]

# 3. View Graph Structure
print("\n--- Graph Structure ---")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of edges: {data.num_edges}")
print(f"Is undirected: {data.is_undirected()}")
print(f"Has isolated nodes: {data.has_isolated_nodes()}")
print(f"Has self-loops: {data.has_self_loops()}")

# 4. Print one Node's Features
# Nodes are indexed. Let's look at node 0.
print("\n--- Node 0 Inspection ---")
print(f"Features (first 5 values): {data.x[0][:5]}")
print(f"Label: {data.y[0].item()}")

# 5. Inspect the Edge Index
# This is a [2, num_edges] tensor representing (source, target)
print("\n--- Edge Index ---")
print(data.edge_index)

Number of graphs: 1
Number of classes: 40
Number of node features: 128

--- Graph Structure ---
Number of nodes: 169343
Number of edges: 2315598
Is undirected: True
Has isolated nodes: False
Has self-loops: False

--- Node 0 Inspection ---
Features (first 5 values): tensor([-0.0579, -0.0525, -0.0726, -0.0266,  0.1304])
Label: 4

--- Edge Index ---
tensor([[     0,      0,      0,  ..., 169341, 169342, 169342],
        [   411,    640,   1162,  ..., 163274,  27824, 158981]])


In [5]:
split_idx = dataset.get_idx_split()
train_idx, valid_idx, test_idx = split_idx["train"], split_idx["valid"], split_idx["test"]

print(f"\nTraining nodes: {len(train_idx)}")
print(f"Validation nodes: {len(valid_idx)}")
print(f"Test nodes: {len(test_idx)}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)


Training nodes: 90941
Validation nodes: 29799
Test nodes: 48603


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv

class VirtualNodeLayer(torch.nn.Module):
    def __init__(self, hidden_channels, dropout):
        super().__init__()
        # Switched BatchNorm1d to LayerNorm
        self.mlp = nn.Sequential(
            nn.Linear(hidden_channels, 2 * hidden_channels),
            nn.LayerNorm(2 * hidden_channels), 
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(2 * hidden_channels, hidden_channels),
            nn.LayerNorm(hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

    def forward(self, x, vn_x):
        # 1. Update node features with virtual node context
        x = x + vn_x
        
        # 2. Update virtual node feature by aggregating all nodes
        # Mean pooling works well for arxiv's single-graph structure
        vn_x_temp = torch.mean(x, dim=0, keepdim=True) + vn_x
        vn_x = self.mlp(vn_x_temp)
        
        return x, vn_x

In [7]:
# Reversible Block Implementation
class RevGATBlock(nn.Module):
    def __init__(self, in_channels, out_channels, heads, dropout):
        super().__init__()
        self.norm = nn.BatchNorm1d(in_channels)
        self.conv = GATConv(in_channels, out_channels // heads, heads=heads, dropout=dropout)
        
    def forward(self, x, edge_index):
        out = self.norm(x)
        out = F.relu(out)
        out = self.conv(out, edge_index)
        return out

class RevGAT_VN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers, heads, dropout):
        super().__init__()
        self.lin_in = nn.Linear(in_channels, hidden_channels)
        
        # Initial Virtual Node embedding
        self.vn_embedding = nn.Embedding(1, hidden_channels)
        
        self.layers = nn.ModuleList()
        self.vn_layers = nn.ModuleList()
        
        for _ in range(num_layers):
            self.layers.append(RevGATBlock(hidden_channels // 2, hidden_channels // 2, heads, dropout))
            self.vn_layers.append(VirtualNodeLayer(hidden_channels, dropout))
            
        self.lin_out = nn.Linear(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.lin_in(x)
        
        # Initialize VN state
        vn_x = self.vn_embedding(torch.tensor([0], device=x.device))
        
        # Reversible blocks
        x1, x2 = torch.chunk(x, 2, dim=-1)
        
        for i in range(len(self.layers)):
            # 1. Standard RevGAT update
            new_x1 = x1 + self.layers[i](x2, edge_index)
            x1, x2 = x2, new_x1
            
            # 2. Virtual Node Global update
            # Re-combine temporarily for global context
            x_combined = torch.cat([x1, x2], dim=-1)
            x_combined, vn_x = self.vn_layers[i](x_combined, vn_x)
            
            # Re-split for the next reversible layer
            x1, x2 = torch.chunk(x_combined, 2, dim=-1)
            
        x = torch.cat([x1, x2], dim=-1)
        x = F.relu(x)
        return self.lin_out(x).log_softmax(dim=-1)

In [8]:
import torch
import numpy as np
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Configuration
num_runs = 10
epochs = 500
eval_steps = 10 
all_test_results = []

global_best_test_acc = 0.0
global_save_path = 'overall_best_revgat_model.pt'

# Using Label Smoothing to bridge the Val/Test gap
criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

for run in range(num_runs):
    seed = run 
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    
    print(f"\n>>> Starting Run {run+1}/{num_runs} (Seed: {seed})")
    
    # 1. Model init with RevGAT architecture
    # Note: in_channels=768 for RoBERTa
    model = RevGAT_VN(
        in_channels=dataset.num_node_features,
        hidden_channels=256, 
        out_channels=dataset.num_classes,
        num_layers=4,
        heads=4,
        dropout=0.5
    ).to(device)

    # 2. Optimizer and Scheduler
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
    
    # mode='max' because we want to maximize validation accuracy
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10, min_lr=1e-5)
    
    best_run_val_acc = 0
    best_run_test_acc = 0

    # 3. Training Loop
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = criterion(out[train_idx], data.y[train_idx].squeeze())
        loss.backward()
        
        # Adding Gradient Clipping for RevGAT stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()

        # Check performance more frequently
        if epoch % eval_steps == 0:
            model.eval()
            with torch.no_grad():
                logits = model(data.x, data.edge_index)
                y_pred = logits.argmax(dim=-1, keepdim=True)
                
                val_acc = (y_pred[valid_idx] == data.y[valid_idx]).sum().item() / valid_idx.size(0)
                test_acc = (y_pred[test_idx] == data.y[test_idx]).sum().item() / test_idx.size(0)

                # Step the scheduler based on validation accuracy
                scheduler.step(val_acc)
                
                # Log current learning rate
                current_lr = optimizer.param_groups[0]['lr']

                if epoch % 10 == 0:
                    print(f"Epoch {epoch:03d}: Val: {val_acc:.4f}, Test: {test_acc:.4f}, LR: {current_lr:.6f}")
                
                # Best model tracking
                if val_acc > best_run_val_acc:
                    best_run_val_acc = val_acc
                    best_run_test_acc = test_acc

                    if test_acc > global_best_test_acc:
                        global_best_test_acc = test_acc
                        torch.save(model.state_dict(), global_save_path)

    all_test_results.append(best_run_test_acc)
    print(f"Run {run+1} Finished. Best Test Acc for this run: {best_run_test_acc:.4f}")

# 4. Final Reporting
mean_acc = np.mean(all_test_results)
std_acc = np.std(all_test_results)

print("\n" + "="*30)
print(f"FINAL METRICS OVER {num_runs} RUNS")
print(f"Test Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
print("="*30)


>>> Starting Run 1/10 (Seed: 0)
Epoch 010: Val: 0.0416, Test: 0.0303, LR: 0.005000
Epoch 020: Val: 0.1559, Test: 0.1354, LR: 0.005000
Epoch 030: Val: 0.4771, Test: 0.4349, LR: 0.005000
Epoch 040: Val: 0.6362, Test: 0.6190, LR: 0.005000
Epoch 050: Val: 0.6248, Test: 0.6185, LR: 0.005000
Epoch 060: Val: 0.6443, Test: 0.6521, LR: 0.005000
Epoch 070: Val: 0.6877, Test: 0.6831, LR: 0.005000
Epoch 080: Val: 0.6981, Test: 0.6930, LR: 0.005000
Epoch 090: Val: 0.6923, Test: 0.6870, LR: 0.005000
Epoch 100: Val: 0.7032, Test: 0.6998, LR: 0.005000
Epoch 110: Val: 0.6785, Test: 0.6488, LR: 0.005000
Epoch 120: Val: 0.7059, Test: 0.6970, LR: 0.005000
Epoch 130: Val: 0.7092, Test: 0.7020, LR: 0.005000
Epoch 140: Val: 0.7079, Test: 0.7085, LR: 0.005000
Epoch 150: Val: 0.6901, Test: 0.7016, LR: 0.005000
Epoch 160: Val: 0.6978, Test: 0.6893, LR: 0.005000
Epoch 170: Val: 0.7044, Test: 0.6778, LR: 0.005000
Epoch 180: Val: 0.7012, Test: 0.6759, LR: 0.005000
Epoch 190: Val: 0.7171, Test: 0.7046, LR: 0.00500

In [9]:
import os
import pandas as pd
import torch
from ogb.nodeproppred import PygNodePropPredDataset

# 1. Load the graph
dataset = PygNodePropPredDataset(name='ogbn-arxiv')
data = dataset[0]

# 2. Load Mapping Files
# The 'nodeidx2paperid' file maps: Node Index -> MAG Paper ID
path = os.path.join(dataset.root, 'mapping', 'nodeidx2paperid.csv.gz')
nodeidx2paperid = pd.read_csv(path)

# The 'titleabs' file contains: MAG Paper ID -> Title/Abstract
if not os.path.exists('titleabs.tsv'):
    !wget https://snap.stanford.edu/ogb/data/misc/ogbn_arxiv/titleabs.tsv.gz
    !gunzip titleabs.tsv.gz

titleabs = pd.read_csv('titleabs.tsv', sep='\t', names=['paper id', 'title', 'abstract'])

nodeidx2paperid['paper id'] = pd.to_numeric(nodeidx2paperid['paper id'], errors='coerce')
titleabs['paper id'] = pd.to_numeric(titleabs['paper id'], errors='coerce')
nodeidx2paperid = nodeidx2paperid.dropna(subset=['paper id']).astype({'paper id': 'int64'})
titleabs = titleabs.dropna(subset=['paper id']).astype({'paper id': 'int64'})

print("Aligning text to node indices...")
# We merge on nodeidx2paperid so the final dataframe order matches the Graph Node Indices (0 to 169,342)
df = pd.merge(nodeidx2paperid, titleabs, on='paper id', how='left')

# Handle missing text (if any papers in the graph aren't in the tsv)
df['title'] = df['title'].fillna('Unknown Title')
df['abstract'] = df['abstract'].fillna('Unknown Abstract')
df['text'] = df['title'] + ". " + df['abstract']

texts = df['text'].tolist()
print(f"Successfully aligned {len(texts)} texts.")

/usr/local/lib/python3.12/dist-packages/ogb/nodeproppred/dataset_pyg.py:69: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  self.data, self.slices = torch.load(self.processed_paths[0])


--2026-04-22 18:29:34--  https://snap.stanford.edu/ogb/data/misc/ogbn_arxiv/titleabs.tsv.gz
Resolving snap.stanford.edu (snap.stanford.edu)... 171.64.75.80
Connecting to snap.stanford.edu (snap.stanford.edu)|171.64.75.80|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 70213527 (67M) [application/x-gzip]
Saving to: ‘titleabs.tsv.gz’

titleabs.tsv.gz     100%[===================>]  66.96M  20.2MB/s    in 3.3s    

2026-04-22 18:29:37 (20.2 MB/s) - ‘titleabs.tsv.gz’ saved [70213527/70213527]

Aligning text to node indices...
Successfully aligned 169343 texts.


In [10]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Prepare Graph
data.edge_index = to_undirected(data.edge_index)
data.x = torch.load('/kaggle/input/datasets/swetha344/ogbn-arxiv-roberta-embeddings/arxiv_roberta_features.pt').to(torch.float)
data = data.to(device)

split_idx = dataset.get_idx_split()
train_idx = split_idx['train'].to(device)
valid_idx = split_idx['valid'].to(device)
test_idx = split_idx['test'].to(device)

/tmp/ipykernel_55/3115342646.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  data.x = torch.load('/kaggle/input/datasets/swetha344/ogbn-arxiv-roberta-embeddings/arxiv_roberta_features.pt').to(torch.float)


In [13]:
import torch
import numpy as np
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Configuration
num_runs = 10
epochs = 500
eval_steps = 10 
all_test_results = []

global_best_test_acc = 0.0
global_save_path = 'overall_best_revgat_model.pt'

# Using Label Smoothing to bridge the Val/Test gap
criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

for run in range(num_runs):
    seed = run 
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    
    print(f"\n>>> Starting Run {run+1}/{num_runs} (Seed: {seed})")
    
    # 1. Model init with RevGAT architecture
    # Note: in_channels=768 for RoBERTa
    model = RevGAT_VN(
        in_channels=768,
        hidden_channels=256, 
        out_channels=dataset.num_classes,
        num_layers=4,
        heads=4,
        dropout=0.5
    ).to(device)

    # 2. Optimizer and Scheduler
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003, weight_decay=1e-4)
    
    # mode='max' because we want to maximize validation accuracy
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10, min_lr=1e-5)
    
    best_run_val_acc = 0
    best_run_test_acc = 0

    # 3. Training Loop
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = criterion(out[train_idx], data.y[train_idx].squeeze())
        loss.backward()
        
        # Adding Gradient Clipping for RevGAT stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()

        # Check performance more frequently
        if epoch % eval_steps == 0:
            model.eval()
            with torch.no_grad():
                logits = model(data.x, data.edge_index)
                y_pred = logits.argmax(dim=-1, keepdim=True)
                
                val_acc = (y_pred[valid_idx] == data.y[valid_idx]).sum().item() / valid_idx.size(0)
                test_acc = (y_pred[test_idx] == data.y[test_idx]).sum().item() / test_idx.size(0)

                # Step the scheduler based on validation accuracy
                scheduler.step(val_acc)
                
                # Log current learning rate
                current_lr = optimizer.param_groups[0]['lr']

                if epoch % 10 == 0:
                    print(f"Epoch {epoch:03d}: Val: {val_acc:.4f}, Test: {test_acc:.4f}, LR: {current_lr:.6f}")
                
                # Best model tracking
                if val_acc > best_run_val_acc:
                    best_run_val_acc = val_acc
                    best_run_test_acc = test_acc

                    if test_acc > global_best_test_acc:
                        global_best_test_acc = test_acc
                        torch.save(model.state_dict(), global_save_path)

    all_test_results.append(best_run_test_acc)
    print(f"Run {run+1} Finished. Best Test Acc for this run: {best_run_test_acc:.4f}")

# 4. Final Reporting
mean_acc = np.mean(all_test_results)
std_acc = np.std(all_test_results)

print("\n" + "="*30)
print(f"FINAL METRICS OVER {num_runs} RUNS")
print(f"Test Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
print("="*30)


>>> Starting Run 1/10 (Seed: 0)
Epoch 010: Val: 0.2429, Test: 0.2338, LR: 0.003000
Epoch 020: Val: 0.4343, Test: 0.4607, LR: 0.003000
Epoch 030: Val: 0.5717, Test: 0.5680, LR: 0.003000
Epoch 040: Val: 0.6060, Test: 0.6113, LR: 0.003000
Epoch 050: Val: 0.5711, Test: 0.5424, LR: 0.003000
Epoch 060: Val: 0.6732, Test: 0.6708, LR: 0.003000
Epoch 070: Val: 0.6853, Test: 0.6869, LR: 0.003000
Epoch 080: Val: 0.6922, Test: 0.6663, LR: 0.003000
Epoch 090: Val: 0.7214, Test: 0.7086, LR: 0.003000
Epoch 100: Val: 0.6734, Test: 0.6420, LR: 0.003000
Epoch 110: Val: 0.7117, Test: 0.7116, LR: 0.003000
Epoch 120: Val: 0.6441, Test: 0.6605, LR: 0.003000
Epoch 130: Val: 0.7149, Test: 0.7101, LR: 0.003000
Epoch 140: Val: 0.6869, Test: 0.6509, LR: 0.003000
Epoch 150: Val: 0.6831, Test: 0.6502, LR: 0.003000
Epoch 160: Val: 0.6964, Test: 0.7005, LR: 0.003000
Epoch 170: Val: 0.7016, Test: 0.6818, LR: 0.003000
Epoch 180: Val: 0.7087, Test: 0.6940, LR: 0.003000
Epoch 190: Val: 0.7133, Test: 0.6909, LR: 0.00300